# Multi-output BNN + Çok Amaçlı Bayesçi Optimizasyon

Bu notebook, aynı proses ayarının birden fazla operasyonel çıktıyı aynı anda etkilediği bir endüstri mühendisliği problemini ele alır:

- **kalite**: artırmak istiyoruz,
- **enerji tüketimi**: azaltmak istiyoruz,
- **çevrim süresi**: azaltmak istiyoruz.

Tek bir "en iyi" nokta yerine **Pareto cephesi** aranır.

Akış:

```text
proses deneyleri
      ↓
shared multi-output BNN
      ↓
3 çıktının ortak posterioru
      ↓
kalite ↑, -enerji ↑, -çevrim süresi ↑
      ↓
Pareto front + hypervolume
      ↓
BoTorch qLogEHVI
      ↓
yeni deney noktası
```

Bu örnekte üç çıktı aynı Bayesçi gizli katmanı paylaşır. Böylece model, çıktıları üç tamamen bağımsız ağ olarak değil, ortak latent temsil üzerinden öğrenir.

> Not: Likelihood koşullu olarak çıktı bazında faktörize edilmiştir. Çıktılar arasındaki bağımlılık ortak Bayesçi ağırlıklar üzerinden taşınır; tam bir multivariate residual covariance modeli kurulmamıştır.


In [ ]:
# Gerekirse:
# %pip install torch pyro-ppl botorch numpy pandas matplotlib

from typing import Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch import Tensor

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample

from botorch.models.ensemble import EnsembleModel
from botorch.acquisition.multi_objective.logei import (
    qLogExpectedHypervolumeImprovement,
)
from botorch.optim import optimize_acqf
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.utils.multi_objective.hypervolume import Hypervolume

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)
torch.set_default_dtype(torch.float64)


## 1. Sentetik çok çıktılı proses

İki normalize proses parametresi olsun:

\[
x=(x_1,x_2)\in[0,1]^2.
\]

Gerçek sistem üç fiziksel çıktı üretiyor:

\[
y(x)=
\begin{bmatrix}
\text{quality}(x)\\
\text{energy}(x)\\
\text{cycle}(x)
\end{bmatrix}.
\]

Bu fonksiyonlar yalnızca sentetik deney üretmek için biliniyor. Gerçek projede bunların yerine fiziksel deney, MES verisi, simülasyon veya dijital ikiz çağrısı gelir.


In [ ]:
def true_process_physical(X: Tensor) -> Tensor:
    x1 = X[..., 0]
    x2 = X[..., 1]

    quality = (
        72.0
        + 20.0 * torch.exp(-18.0 * ((x1 - 0.68)**2 + (x2 - 0.34)**2))
        + 7.0 * torch.exp(-30.0 * ((x1 - 0.30)**2 + (x2 - 0.76)**2))
        + 2.5 * torch.sin(6.0 * x1) * torch.cos(5.0 * x2)
    )

    energy = (
        38.0
        + 22.0 * x1
        + 14.0 * x2
        + 8.0 * (x1 - x2)**2
        - 5.0 * torch.exp(-20.0 * ((x1 - 0.25)**2 + (x2 - 0.25)**2))
    )

    cycle = (
        54.0
        - 18.0 * x1
        - 10.0 * x2
        + 13.0 * (x1 - 0.60)**2
        + 8.0 * (x2 - 0.55)**2
        + 2.0 * torch.sin(5.0 * x2)
    )

    return torch.stack([quality, energy, cycle], dim=-1)


def observe_physical(X: Tensor) -> Tensor:
    y = true_process_physical(X)
    noise_sd = torch.tensor([0.8, 0.7, 0.6], dtype=X.dtype)
    return y + torch.randn_like(y) * noise_sd


# BoTorch varsayılan olarak bütün objectives'i maksimize eder.
# Bu yüzden minimize edilen enerji ve çevrim süresinin işaretini çeviriyoruz.
def to_maximization_objectives(Y_phys: Tensor) -> Tensor:
    return torch.stack(
        [Y_phys[..., 0], -Y_phys[..., 1], -Y_phys[..., 2]],
        dim=-1,
    )


bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])

sobol = torch.quasirandom.SobolEngine(2, scramble=True, seed=SEED)
train_X = sobol.draw(24)
train_Y_phys = observe_physical(train_X)
train_Z = to_maximization_objectives(train_Y_phys)

pd.DataFrame(
    torch.cat([train_X, train_Y_phys], dim=-1).numpy(),
    columns=["x1", "x2", "quality", "energy", "cycle"],
).head()


## 2. Pareto üstünlüğü

Bir çözüm \(a\), \(b\)'yi ancak bütün amaçlarda en az onun kadar iyi ve en az bir amaçta daha iyi ise domine eder.

Üç amaçlı durumda tek bir optimum yerine:

\[
\mathcal P=
\{x:\nexists x' \text{ that dominates }x\}
\]

kümesi aranır.

Enerji ve çevrim süresinin işaretini çevirdiğimiz için bütün objective uzayında "daha büyük daha iyi" kuralını kullanabiliriz.


In [ ]:
pareto_mask = is_non_dominated(train_Z)
pareto_phys = train_Y_phys[pareto_mask]
pareto_X = train_X[pareto_mask]

print("Başlangıç deney sayısı:", len(train_X))
print("Başlangıç Pareto noktası sayısı:", int(pareto_mask.sum()))

plt.figure(figsize=(7, 5))
plt.scatter(
    train_Y_phys[:, 1].numpy(),
    train_Y_phys[:, 0].numpy(),
    label="Deneyler",
    alpha=0.65,
)
plt.scatter(
    pareto_phys[:, 1].numpy(),
    pareto_phys[:, 0].numpy(),
    marker="x",
    s=90,
    label="Pareto (3 amaç dikkate alınarak)",
)
plt.xlabel("Enerji (az iyi)")
plt.ylabel("Kalite (çok iyi)")
plt.legend()
plt.title("Kalite–enerji görünümü")
plt.show()


## 3. Çok çıktılı BNN

Ortak Bayesçi gizli katman:

\[
h_w(x)=\tanh(W_1x+b_1)
\]

ve üç çıktılı Bayesçi başlık:

\[
\mu_w(x)=W_2 h_w(x)+b_2
\in\mathbb R^3.
\]

Likelihood:

\[
Y\mid x,w
\sim
\mathcal N(
\mu_w(x),
\operatorname{diag}(\sigma_1^2,\sigma_2^2,\sigma_3^2)
).
\]

Burada residual covariance diagonal; fakat posterior objective örnekleri ortak Bayesçi ağırlıkları paylaştığı için tamamen bağımsız değildir.


In [ ]:
# BNN eğitimini numerik olarak kolaylaştırmak için objectives'i standardize ediyoruz.
Z_mean = train_Z.mean(dim=0)
Z_std = train_Z.std(dim=0).clamp_min(1e-6)
train_Y = (train_Z - Z_mean) / Z_std


class MultiOutputBNN(PyroModule):
    MU_SITE = "mu"

    def __init__(self, in_features=2, hidden=24, outputs=3):
        super().__init__()
        self.outputs = outputs

        self.hidden = PyroModule[nn.Linear](in_features, hidden)
        self.hidden.weight = PyroSample(
            dist.Normal(0.0, 0.9)
            .expand([hidden, in_features])
            .to_event(2)
        )
        self.hidden.bias = PyroSample(
            dist.Normal(0.0, 0.9)
            .expand([hidden])
            .to_event(1)
        )

        self.out = PyroModule[nn.Linear](hidden, outputs)
        self.out.weight = PyroSample(
            dist.Normal(0.0, 0.8)
            .expand([outputs, hidden])
            .to_event(2)
        )
        self.out.bias = PyroSample(
            dist.Normal(0.0, 0.8)
            .expand([outputs])
            .to_event(1)
        )

    def forward(
        self,
        X: Tensor,
        Y: Optional[Tensor] = None,
    ) -> Tensor:
        # Acquisition optimizasyonunda X'e göre gradient gerekir.
        torch.set_grad_enabled(True)

        h = torch.tanh(self.hidden(X))
        mu = self.out(h)
        pyro.deterministic(self.MU_SITE, mu)

        sigma = pyro.sample(
            "sigma",
            dist.LogNormal(-2.2, 0.35)
            .expand([self.outputs])
            .to_event(1),
        )

        with pyro.plate("data", X.shape[0]):
            pyro.sample(
                "obs",
                dist.Normal(mu, sigma).to_event(1),
                obs=Y,
            )

        return mu


pyro.clear_param_store()
multi_bnn = MultiOutputBNN()
guide = AutoDiagonalNormal(multi_bnn)

svi = SVI(
    multi_bnn,
    guide,
    pyro.optim.Adam({"lr": 0.015}),
    loss=Trace_ELBO(),
)

losses = []
for step in range(2200):
    losses.append(svi.step(train_X, train_Y) / len(train_X))
    if (step + 1) % 550 == 0:
        print(f"Adım {step+1}: ELBO/gözlem={losses[-1]:.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("SVI adımı")
plt.ylabel("ELBO / gözlem")
plt.show()


## 4. Pyro posteriorunu BoTorch multi-output modeline bağlama

BoTorch'un Monte Carlo acquisition fonksiyonları GP zorunluluğu koymaz. Modelin örneklenebilir bir posterior üretmesi yeterlidir.

Wrapper aşağıdaki şekli üretir:

```text
batch_shape × posterior_sample × q × 3 objectives
```

BoTorch 0.18.x'te `EnsembleModel` bu amaç için kullanılabilir.


In [ ]:
class PyroMultiOutputBoTorchModel(EnsembleModel):
    _num_outputs = 3

    def __init__(self, pyro_model, guide, num_samples=128):
        super().__init__()
        self.pyro_model = pyro_model
        self.guide = guide
        self.num_samples = num_samples

    def forward(self, X: Tensor) -> Tensor:
        # X: batch_shape x q x d
        batch_shape = X.shape[:-2]
        q = X.shape[-2]
        d = X.shape[-1]

        X_flat = X.reshape(-1, d)

        predictive = Predictive(
            self.pyro_model,
            guide=self.guide,
            num_samples=self.num_samples,
            return_sites=(self.pyro_model.MU_SITE,),
        )

        # s x (prod(batch_shape)*q) x m
        raw = predictive(X_flat)[self.pyro_model.MU_SITE]

        # s x batch_shape... x q x m
        raw = raw.reshape(
            self.num_samples,
            *batch_shape,
            q,
            self._num_outputs,
        )

        # batch_shape... x s x q x m
        k = len(batch_shape)
        perm = list(range(1, 1 + k)) + [0, k + 1, k + 2]
        return raw.permute(*perm)


botorch_bnn = PyroMultiOutputBoTorchModel(
    multi_bnn,
    guide,
    num_samples=128,
)

with torch.no_grad():
    test_post = botorch_bnn.posterior(train_X[:4].unsqueeze(-2))

print("Posterior mean shape:", tuple(test_post.mean.shape))
print("Posterior variance shape:", tuple(test_post.variance.shape))


## 5. Posterior objective yüzeyleri

Posterior standardize objective uzayındadır:

1. kalite,
2. \(-\)enerji,
3. \(-\)çevrim süresi.

Fiziksel yorum için posterior ortalamasını tekrar orijinal ölçeğe çeviriyoruz.


In [ ]:
grid_n = 55
g1 = torch.linspace(0, 1, grid_n)
g2 = torch.linspace(0, 1, grid_n)
G1, G2 = torch.meshgrid(g1, g2, indexing="ij")
grid_X = torch.stack(
    [G1.reshape(-1), G2.reshape(-1)],
    dim=-1,
)

with torch.no_grad():
    grid_post = botorch_bnn.posterior(grid_X.unsqueeze(-2))
    grid_mean_std = grid_post.mean.squeeze(-2)
    grid_sd_std = grid_post.variance.sqrt().squeeze(-2)

grid_mean_Z = grid_mean_std * Z_std + Z_mean
grid_mean_phys = torch.stack(
    [
        grid_mean_Z[:, 0],
        -grid_mean_Z[:, 1],
        -grid_mean_Z[:, 2],
    ],
    dim=-1,
)

for j, title in enumerate(
    ["Kalite posterior ortalaması", "Enerji posterior ortalaması", "Çevrim süresi posterior ortalaması"]
):
    plt.figure(figsize=(6, 4.7))
    surface = grid_mean_phys[:, j].reshape(grid_n, grid_n)
    plt.contourf(G1.numpy(), G2.numpy(), surface.numpy(), levels=25)
    plt.scatter(train_X[:, 0].numpy(), train_X[:, 1].numpy(), marker="x", s=25)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.show()


## 6. Hypervolume

Pareto cephesinin kalitesini tek sayıyla izlemek için **hypervolume** kullanılabilir.

Bir reference point \(r\), ilgi alanındaki bütün kabul edilebilir objective değerlerinden daha kötü seçilir. Maksimizasyon probleminde Pareto cephesinin \(r\)'ye göre domine ettiği hacim ölçülür.

Bu referans noktası bir teorem tarafından otomatik belirlenmez; domain bilgisiyle sabitlenmesi tercih edilir. Öğretim örneğinde ilk veri setinin altında kalacak şekilde seçiyoruz.


In [ ]:
# Standardize objective uzayında sabit reference point.
# Başlangıç veri setinden daha kötü bir nokta seçiyoruz ve BO boyunca sabit tutuyoruz.
ref_point = train_Y.min(dim=0).values - 0.35

initial_mask = is_non_dominated(train_Y)
initial_pareto = train_Y[initial_mask]

hv = Hypervolume(ref_point=ref_point)
initial_hv = float(hv.compute(initial_pareto))

print("Reference point:", ref_point.numpy().round(3))
print("Başlangıç hypervolume:", round(initial_hv, 4))


## 7. qLogEHVI ile yeni deney seçme

BoTorch 0.18.1, multi-objective BO için `qLogExpectedHypervolumeImprovement` (qLogEHVI) ve gürültülü problemler için `qLogNoisyExpectedHypervolumeImprovement` (qLogNEHVI) sağlar.

Burada basit öğretim örneğinde qLogEHVI kullanıyoruz.

\[
x_{\text{next}}
=
\arg\max_x
\operatorname{qLogEHVI}(x).
\]

Acquisition yeni noktanın Pareto cephesinin hypervolume'unu ne kadar artırabileceğini posterior belirsizliği altında değerlendirir.


In [ ]:
partitioning = FastNondominatedPartitioning(
    ref_point=ref_point,
    Y=train_Y,
)

acq = qLogExpectedHypervolumeImprovement(
    model=botorch_bnn,
    ref_point=ref_point.tolist(),
    partitioning=partitioning,
)

candidate, acq_value = optimize_acqf(
    acq_function=acq,
    bounds=bounds,
    q=1,
    num_restarts=8,
    raw_samples=128,
    options={"batch_limit": 4, "maxiter": 150},
)

candidate = candidate.detach()
new_Y_phys = observe_physical(candidate)
new_Z = to_maximization_objectives(new_Y_phys)
new_Y = (new_Z - Z_mean) / Z_std

print("Önerilen proses ayarı:", candidate.numpy().round(4))
print(
    pd.DataFrame(
        new_Y_phys.numpy(),
        columns=["quality", "energy", "cycle"],
    )
)
print("qLogEHVI değeri:", float(acq_value))


## 8. Yeni deney Pareto cephesini geliştirdi mi?

Multi-objective optimizasyonda "tek objective iyileşti mi?" sorusu yeterli değildir.

Kontrol edilebilecek metrikler:

- non-dominated nokta sayısı,
- hypervolume,
- Pareto trade-off'larının operasyonel anlamı,
- gerçek deney bütçesi başına hypervolume artışı.


In [ ]:
Y_aug = torch.cat([train_Y, new_Y], dim=0)
Y_phys_aug = torch.cat([train_Y_phys, new_Y_phys], dim=0)

pareto_aug_mask = is_non_dominated(Y_aug)
pareto_aug = Y_aug[pareto_aug_mask]
pareto_phys_aug = Y_phys_aug[pareto_aug_mask]

new_hv = float(hv.compute(pareto_aug))

print("Başlangıç HV:", round(initial_hv, 4))
print("Yeni HV:", round(new_hv, 4))
print("HV değişimi:", round(new_hv - initial_hv, 4))
print("Pareto nokta sayısı:", len(pareto_aug))

plt.figure(figsize=(7, 5))
plt.scatter(
    Y_phys_aug[:, 1].numpy(),
    Y_phys_aug[:, 0].numpy(),
    alpha=0.45,
    label="Tüm gözlemler",
)
plt.scatter(
    pareto_phys_aug[:, 1].numpy(),
    pareto_phys_aug[:, 0].numpy(),
    marker="x",
    s=90,
    label="3 amaçlı Pareto kümesi",
)
plt.scatter(
    new_Y_phys[:, 1].numpy(),
    new_Y_phys[:, 0].numpy(),
    marker="*",
    s=180,
    label="Yeni qLogEHVI deneyi",
)
plt.xlabel("Enerji (az iyi)")
plt.ylabel("Kalite (çok iyi)")
plt.legend()
plt.title("Yeni deney sonrası Pareto görünümü")
plt.show()


## 9. Endüstri mühendisliği açısından ne öğrendik?

Bu mimari aşağıdaki tipte problemlere doğrudan uyarlanabilir:

| Karar | Objective 1 | Objective 2 | Objective 3 |
|---|---|---|---|
| Üretim prosesi | kalite ↑ | enerji ↓ | çevrim süresi ↓ |
| Bakım politikası | availability ↑ | bakım maliyeti ↓ | arıza riski ↓ |
| Tedarik zinciri | servis seviyesi ↑ | maliyet ↓ | CO₂ ↓ |
| Çizelgeleme | throughput ↑ | tardiness ↓ | enerji ↓ |
| Tasarım optimizasyonu | performans ↑ | ağırlık ↓ | maliyet ↓ |

### Kritik nüanslar

1. **Multi-output** ile **multi-objective** aynı şey değildir.  
   Multi-output BNN birden fazla rassal çıktıyı modeller; multi-objective optimization bu çıktılardan hangilerinin nasıl tercih edildiğini tanımlar.

2. Bu notebook residual covariance'i diagonal varsayar.  
   Ortak Bayesian backbone objective'ler arasında posterior bağımlılık yaratır, fakat tam korelasyonlu çok değişkenli likelihood daha zengin bir modeldir.

3. Pareto cephesi "tek doğru karar" vermez.  
   Son karar için yönetim tercihi, maliyet, servis seviyesi veya kısıtlar gerekir.

4. Hypervolume reference point operasyonel bir modelleme tercihidir.  
   Reference point değiştirilirse hypervolume değeri değişebilir.

5. Gerçek projede BNN mutlaka GP, Deep Ensemble ve mümkünse klasik response-surface modelleriyle karşılaştırılmalıdır.

### İleri genişletmeler

- qLogNEHVI ile gürültülü MOBO,
- constrained MOBO,
- correlated multivariate likelihood,
- multi-task/hierarchical BNN,
- risk-aware multi-objective acquisition,
- Ax ile experiment orchestration.
